# Finance & Accounting Analytics with Python

This notebook demonstrates a practical workflow using synthetic finance transactions. It covers data quality, budget variance, vendor spend and exception testing.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

df = pd.read_csv(Path('../data/sample_transactions.csv'))
df.head()

## 1. Data quality checks

In [ ]:
df.info()
print('Missing values:')
print(df.isna().sum())
print('Duplicate invoice numbers:')
print(df[df.duplicated('invoice_no', keep=False)][['transaction_id','invoice_no','vendor','amount']])

## 2. Budget versus actual

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.to_period('M').astype(str)
monthly = df.groupby('month', as_index=False).agg(budget=('budget','sum'), actual=('amount','sum'))
monthly['variance'] = monthly['actual'] - monthly['budget']
monthly['variance_pct'] = monthly['variance'] / monthly['budget'] * 100
monthly

In [ ]:
ax = monthly.plot(x='month', y=['budget','actual'], kind='bar', figsize=(9,5), title='Monthly Budget vs Actual')
ax.set_ylabel('Amount')
plt.tight_layout()
plt.show()

## 3. Vendor concentration

In [ ]:
vendor = df.groupby('vendor', as_index=False).agg(transactions=('transaction_id','count'), spend=('amount','sum')).sort_values('spend', ascending=False)
vendor

## 4. Exception testing

The following tests are designed as practical audit analytics examples: duplicate invoices, material budget overruns and high-value transactions.

In [ ]:
df['variance_pct'] = (df['amount'] - df['budget']) / df['budget'] * 100
overruns = df[df['variance_pct'] > 10].sort_values('variance_pct', ascending=False)
high_value = df[df['amount'] >= 150000].sort_values('amount', ascending=False)
print('Budget overruns > 10%')
display(overruns[['transaction_id','vendor','category','budget','amount','variance_pct']])
print('High-value transactions >= 150,000')
display(high_value[['transaction_id','vendor','amount']])

## Management interpretation

A useful finance analytics workflow does not stop at identifying an exception. Each exception should be assessed for cause, financial impact, control relevance and required management action. The sample dataset intentionally includes a duplicate invoice and several overruns so the testing logic can be demonstrated without using confidential information.